In [1]:
source(here::here("data-cleaning", "00a-parameters.r"))


Parallelization: FALSE 


# Libraries


In [2]:
# Update the grouper
system("git submodule update --init --recursive")

# List required packages
required_packages <- c(
  "data.table", # Fast data manipulation
  "here", # Simplifies file path management
  "tictoc", # Timing code execution
  "stringr", # String manipulation
  "stringi", # Unicode string processing
  "lubridate", # Date-time handling
  "profvis", # Profiling R code
  "hash", # Hashing utility
  "future", # Parallel processing
  "future.apply", # Parallelized apply functions
  "knitr", # Dynamic report generation
  "htmlwidgets", # Interactive HTML widgets
  "parallelly", # Advanced parallel computing
  "stringdist", # String distance calculations
  "parallel", # Base parallel computing
  "reticulate", # Interface to Python
  "bigrquery", # BigQuery client
  "jsonlite", # JSON parsing
  "googleCloudStorageR", # Google Cloud Storage access
  "haven", # Read/write Stata, SPSS, SAS files
  "fst", # Fast serialization
  "httr", # HTTP requests
  "ggplot2", # Data visualization
  "rmarkdown", # Dynamic markdown documents
  "digest", # Create cryptographic hashes
  "base64enc", # Base64 encoding/decoding
  "arrow", # Apache Arrow for fast data storage
  "tidyverse", # Collection of data science packages,
  "fasttime", # for fastPOSIXct
  "glue" # for string pasting
)

github_packages <- c(
  "r-lib/styler" # Code formatting
)

# Installation commands (commented out, for reference)
invisible(lapply(
  required_packages, function(pkg) {
    if (!require(pkg, character.only = TRUE)) {
      install.packages(pkg)
    }
  }
))
invisible(lapply(
  github_packages, function(repo) {
    if (!require(basename(repo), character.only = TRUE)) {
      remotes::install_github(repo)
    }
  }
))

# Load packages (assumes they are already installed)
invisible(lapply(required_packages, library, character.only = TRUE))
invisible(lapply(basename(github_packages), library, character.only = TRUE))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc/drg-pipeline

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future

Loading required package: future.apply

Loading

# R Scripts


In [3]:
year <- 2018

# Source each file sequentially
for (file in list.files(
  here::here("data-cleaning/r_scripts_v2"),
  pattern = "\\.R$", full.names = TRUE
)) {
  invisible(source(file))
}

message(year)


ℹ 2025-02-17 06:31:08.324888 > Setting client.id from options(googleAuthR.client_id)



The following directories were created:


/home/resurreccion_cmc/drg-pipeline/data-cleaning/data/chkpts/chkpt_4_thai_master_input,
/home/resurreccion_cmc/drg-pipeline/data-cleaning/data/chkpts/chkpt_5_thai_output 


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)


2018



In [4]:
to_bq <- TRUE
bq_table <- "phic_claims"
gcp_proj <- "drg-pipeline"
to_sample <- FALSE

# Define directories using here()
pre_dir <- here::here("drg-pipeline", "data-cleaning", "data", "chkpts", "chkpt_4_thai_master_input")
post_dir <- here::here("drg-pipeline", "data-cleaning", "data", "chkpts", "chkpt_5_thai_output")

# Ensure directories exist
dir.create(here::here(pre_dir, "txt_raw"), recursive = TRUE, showWarnings = FALSE)
dir.create(here::here(post_dir, "txt_raw"), recursive = TRUE, showWarnings = FALSE)


In [5]:
# Use paste0() with here() to construct gsutil commands
system(paste0("gsutil -m cp 'gs://phic-claims-checkpoints/pre-tdrg/*.txt' ", shQuote(here::here(pre_dir, "txt_raw"))), intern = TRUE)
system(paste0("gsutil -m cp 'gs://phic-claims-checkpoints/post-tdrg/*.TXT' ", shQuote(here::here(post_dir, "txt_raw"))), intern = TRUE)


In [ ]:
# Function to process files by year
process_files <- function(directory, prefix) {
  file_list <- Sys.glob(here::here(directory, "*.txt"))
  years <- unique(gsub(".*_([0-9]{4})_.*", "\\1", file_list))

  rds_dir <- here::here(directory, "txt_rds")
  dir.create(rds_dir, recursive = TRUE, showWarnings = FALSE)

  for (year in years) {
    year_files <- grep(paste0("_", year, "_"), file_list, value = TRUE)

    if (length(year_files) > 0) {
      dt <- rbindlist(lapply(year_files, fread, na.strings = "--"), use.names = TRUE, fill = TRUE)
      saveRDS(dt, file = here::here(rds_dir, paste0(prefix, year, ".rds")), compress = FALSE)
      print(paste0("Saved: ", prefix, year, ".rds"))
    }
  }
}

# Process pre-tdrg and post-tdrg files
process_files(pre_dir, "pre_tdrg_")
process_files(post_dir, "post_tdrg_")


In [ ]:
library(bigrquery)
library(data.table)

# Set environment variable for BigQuery project
Sys.setenv(BIGQUERY_TEST_PROJECT = "drg-pipeline")
billing <- bq_test_project()

# Define parameters
dataset <- "drg_claims"
chkpt_dir <- here::here("drg-pipeline", "data-cleaning", "data", "chkpts", "chkpt_5_thai_output", "bq_rds")

# Ensure checkpoint directory exists
dir.create(chkpt_dir, recursive = TRUE, showWarnings = FALSE)

# Function to fetch and save BigQuery data in batches, then save as a single file
fetch_and_save_bq_combined <- function(year, batch_size = 100000) {
  table_name <- paste0(dataset, ".thai_", year)
  offset <- 0 # Starting offset
  batch_number <- 1 # Track batch count
  all_data <- list() # Store batches in a list

  while (TRUE) {
    # Construct the SQL query
    sql <- paste0(
      "SELECT * FROM `", billing, ".", table_name, "` ",
      "LIMIT ", as.integer(batch_size), " OFFSET ", as.integer(offset)
    )

    # Debugging: Print SQL Query
    # print(paste0("Executing SQL: ", sql))

    # Run query and load results
    tb <- tryCatch(
      bq_project_query(billing, sql),
      error = function(e) {
        print(paste0("Error querying table: ", table_name, " Batch: ", batch_number))
        return(NULL)
      }
    )

    # Stop if the query failed
    if (is.null(tb)) break

    df <- tryCatch(
      bq_table_download(tb),
      error = function(e) {
        print(paste0("Error downloading data for year ", year, ", batch ", batch_number))
        return(NULL)
      }
    )

    # Break the loop if there are no more rows to fetch
    if (is.null(df) || nrow(df) == 0) break

    # Convert to data.table and add to list
    all_data[[batch_number]] <- as.data.table(df)

    # Update offset and batch number for next iteration
    offset <- offset + batch_size
    batch_number <- batch_number + 1
  }

  # Combine all batches into a single data.table
  if (length(all_data) > 0) {
    final_data <- rbindlist(all_data, use.names = TRUE, fill = TRUE) # Combine batches
    rds_path <- here::here(chkpt_dir, paste0("bq_dt_", year, ".rds"))
    saveRDS(final_data, rds_path, compress = FALSE)
    print(paste0("Saved full dataset for year ", year, " to ", rds_path))
  } else {
    print(paste0("No data retrieved for year ", year))
  }
}

# Fetch and save BigQuery data for years 2018-2023
for (year in 2018:2023) fetch_and_save_bq_combined(as.character(year))
